# 原始 Pressure：STEMNIST_CSNN 训练与最终测试

输入只执行 `/255` 缩放，不做标准化。训练主体与 `STEMNIST_Classify` 保持一致。

In [12]:
import os

# 必须在首次创建 CUDA 上下文前设置，确保 cuBLAS 使用确定性算法。
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import random
from pathlib import Path
import importlib
import sys
import numpy as np

# SpikingJelly 旧版 CuPy 后端仍会访问已被 NumPy 删除的 np.int。
# np.int 原本就是 Python int 的别名，在这里恢复该别名以保持兼容。
if "int" not in np.__dict__:
    np.int = int
import torch
import torch.nn as nn
from spikingjelly.activation_based import functional

In [13]:
def find_project_root():
    # 从 Notebook 当前工作目录逐级向上查找项目根目录。
    current = Path.cwd().resolve()

    for candidate in (current, *current.parents):
        loader_path = candidate / "src" / "data" / "loader.py"

        if loader_path.is_file():
            return candidate

    raise FileNotFoundError("无法找到 STEMNIST_Ready 项目根目录")


PROJECT_ROOT = find_project_root()

# 导入 src.data 时，需要把 src 的父目录加入模块搜索路径。
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 如果文件是在 Notebook 启动后创建的，刷新模块缓存。
importlib.invalidate_caches()

In [14]:
# True 表示优先保证同一环境中多次训练结果可重复。
REPRODUCIBLE = False    # 开启后训练速度变慢很多
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.use_deterministic_algorithms(REPRODUCIBLE)
torch.backends.cudnn.deterministic = REPRODUCIBLE
torch.backends.cudnn.benchmark = not REPRODUCIBLE
torch.set_float32_matmul_precision(
    "highest" if REPRODUCIBLE else "high"
)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = not REPRODUCIBLE
    torch.backends.cudnn.allow_tf32 = not REPRODUCIBLE

## 1. 参数定义

In [15]:
# ========================================================
# 数据参数
# ========================================================

DATA_KIND = "pressure"
BATCH_SIZE = 64
TIME_STEPS = 240
NUM_WORKERS = min(8, os.cpu_count() or 1)
PREFETCH_FACTOR = 4
LOAD_DATA_IN_MEMORY = torch.cuda.is_available()

# AMP 保留 Tensor Core 加速；严格复现时使用 Torch LIF 后端。
AMP_ENABLED = torch.cuda.is_available()
AMP_DTYPE = torch.float16
AMP_INIT_SCALE = 1024.0
SNN_BACKEND = (
    "torch"
    if REPRODUCIBLE
    else ("cupy" if torch.cuda.is_available() else "torch")
)
PROGRESS_UPDATE_INTERVAL = 20

# ========================================================
# 模型参数
# ========================================================

MODEL_NAME = "STEMNIST_CSNN"
DROPOUT_RATE = 0.1
TAU = 10.0
# BN_MOMENTUM = 0.1  # model_v4不使用BN层(使用BN层会导致验证集准确率抖动)

# ========================================================
# 训练参数
# ========================================================

LEARNING_RATE = 0.005
MIN_LEARNING_RATE = 1e-5
WARMUP_EPOCHS = 5
NUM_EPOCHS = 100

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# ========================================================
# 实验名称
# ========================================================

EXPERIMENT_NAME = (
    f"{MODEL_NAME}"
    f"_T{TIME_STEPS}"
    f"_dropout_{DROPOUT_RATE}"
    f"_batchsize_{BATCH_SIZE}"
    f"_lr_{LEARNING_RATE}"
    f"_tau_{TAU}"
    # f"_bnmom_{BN_MOMENTUM}"
    f"_seed_{SEED}"
    f"_det_{int(REPRODUCIBLE)}"
)

# ========================================================
# 输出路径
# ========================================================

OUTPUT_ROOT = PROJECT_ROOT / "outputs"
EXPERIMENT_DIR = OUTPUT_ROOT / DATA_KIND
DATA_OUTPUT_DIR = EXPERIMENT_DIR / "data"
FIGURE_OUTPUT_DIR = EXPERIMENT_DIR / "figure"

BEST_MODEL_PATH = EXPERIMENT_DIR / "best_model.pt"
HISTORY_PLOT_PATH = FIGURE_OUTPUT_DIR / "training_history.png"
HISTORY_CSV_PATH = DATA_OUTPUT_DIR / "history.csv"

CHECKPOINT_METADATA = {
    "data_kind": DATA_KIND,
    "model_name": MODEL_NAME,
    "time_steps": TIME_STEPS,
    "batch_size": BATCH_SIZE,
    "seed": SEED,
    "reproducible": REPRODUCIBLE,
    "backend": SNN_BACKEND,
    "amp_enabled": AMP_ENABLED,
    "amp_dtype": str(AMP_DTYPE),
    "amp_init_scale": AMP_INIT_SCALE,
}

DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("数据类型：", DATA_KIND)
print("严格复现：", REPRODUCIBLE)
print("随机种子：", SEED)
print("实验目录：", EXPERIMENT_DIR)
print("模型路径：", BEST_MODEL_PATH)
print("历史记录图片路径：", HISTORY_PLOT_PATH)
print("历史记录CSV路径：", HISTORY_CSV_PATH)


数据类型： pressure
严格复现： False
随机种子： 42
实验目录： /root/autodl-tmp/Neurophic_System2STEMNIST/outputs/pressure
模型路径： /root/autodl-tmp/Neurophic_System2STEMNIST/outputs/pressure/best_model.pt
历史记录图片路径： /root/autodl-tmp/Neurophic_System2STEMNIST/outputs/pressure/figure/training_history.png
历史记录CSV路径： /root/autodl-tmp/Neurophic_System2STEMNIST/outputs/pressure/data/history.csv


## 2. 数据

In [16]:
from src.data.transform import build_pressure_transform
from src.data.loader import LoaderConfig, create_loaders

In [17]:
# 压力数据转换为 float32，并从 [0, 255] 缩放到 [0, 1]。
pressure_transform = build_pressure_transform()
config = LoaderConfig(
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    seed=SEED,
    prefetch_factor=PREFETCH_FACTOR,
    in_memory=LOAD_DATA_IN_MEMORY,
)

pressure_loaders = create_loaders(
    data_root=PROJECT_ROOT / "data",
    data_kind=DATA_KIND,
    train_transform=pressure_transform,
    eval_transform=pressure_transform,
    config=config,
)
train_loader = pressure_loaders["train"]
val_loader = pressure_loaders["val"]
test_loader = pressure_loaders["test"]

In [18]:
print(f"Train loader length: {len(train_loader)}")
print(f"Validation loader length: {len(val_loader)}")
print(f"Test loader length: {len(test_loader)}")

Train loader length: 85
Validation loader length: 19
Test loader length: 19


## 3. 模型

In [19]:
# 本项目只保留论文最终使用的 v4 模型。

In [20]:
from src.models.stemnist_csnn import STEMNIST_CSNN

model = STEMNIST_CSNN(
    num_classes=35,
    dropout=DROPOUT_RATE,
    tau=TAU,
    logit_scale=1.0,
    temporal_bins=4,
    backend=SNN_BACKEND,
).to(DEVICE)

model.parameter_count()


97027

## 4. 损失优化

In [21]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4,
)

warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=WARMUP_EPOCHS,
)

cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=NUM_EPOCHS - WARMUP_EPOCHS,
    eta_min=MIN_LEARNING_RATE,
)

scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[
        warmup_scheduler,
        cosine_scheduler,
    ],
    milestones=[
        WARMUP_EPOCHS,
    ],
)

## 5. 训练

In [22]:
from src.function_utils import train_epoch, validate_epoch, train_model

In [ ]:
history = train_model(model, 
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=DEVICE,
            num_epochs=NUM_EPOCHS,
            save_path=BEST_MODEL_PATH,
            scheduler=scheduler,
            amp_enabled=AMP_ENABLED,
            amp_dtype=AMP_DTYPE,
            amp_init_scale=AMP_INIT_SCALE,
            progress_update_interval=PROGRESS_UPDATE_INTERVAL,
            checkpoint_metadata=CHECKPOINT_METADATA,
)

Train Epoch 1:   0%|          | 0/85 [00:00<?, ?it/s]

Val Epoch 1:   0%|          | 0/19 [00:00<?, ?it/s]

Validation firing rates | lif1=0.271999 | lif2=0.158353 | output=0.062382

Epoch 001/100 | Train loss: 3.5578 | Train accuracy: 0.0288 | Val loss: 3.5554 | Val accuracy: 0.0286 | LR: 0.0005 | Train: 467.2 samples/s | GPU peak: 9.43 GiB
✓ 保存最佳模型：/root/autodl-tmp/Neurophic_System2STEMNIST/outputs/pressure/best_model.pt
  epoch=1, val_accuracy=0.0286


Train Epoch 2:   0%|          | 0/85 [00:00<?, ?it/s]

Val Epoch 2:   0%|          | 0/19 [00:00<?, ?it/s]

...

Validation firing rates | lif1=0.254475 | lif2=0.192484 | output=0.148712

Epoch 074/100 | Train loss: 3.5557 | Train accuracy: 0.0230 | Val loss: 3.5554 | Val accuracy: 0.0286 | LR: 0.000940195 | Train: 2496.9 samples/s | GPU peak: 4.23 GiB


Train Epoch 75:   0%|          | 0/85 [00:00<?, ?it/s]

Val Epoch 75:   0%|          | 0/19 [00:00<?, ?it/s]

Validation firing rates | lif1=0.254475 | lif2=0.192484 | output=0.148712

Epoch 075/100 | Train loss: 3.5556 | Train accuracy: 0.0286 | Val loss: 3.5554 | Val accuracy: 0.0286 | LR: 0.000876799 | Train: 2715.6 samples/s | GPU peak: 4.23 GiB


Train Epoch 76:   0%|          | 0/85 [00:00<?, ?it/s]

## 6. 结果可视化与数据保存

In [ ]:
from src.function_utils import plot_training_history

In [ ]:
plot_training_history(
    history,
    save_path=HISTORY_PLOT_PATH,
)

In [ ]:
# 保存可重新绘图的完整训练历史。
from src.reporting import save_history_csv

save_history_csv(history, HISTORY_CSV_PATH)


## 最佳验证模型的最终测试

重新载入 100 epochs 中验证准确率最高的 checkpoint，测试集只评估一次。

In [ ]:
# train_model 已载入最佳权重；此处再次显式载入，保证测试来源清晰。
best_checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=False,
)
model.load_state_dict(best_checkpoint["model_state_dict"])
functional.reset_net(model)

from src.evaluation import evaluate_test
from src.reporting import (
    plot_confusion_matrix,
    plot_per_class_accuracy,
    plot_training_and_test_summary,
    save_test_data,
)

test_result = evaluate_test(
    model=model,
    test_loader=test_loader,
    criterion=criterion,
    device=DEVICE,
    amp_enabled=AMP_ENABLED,
    amp_dtype=AMP_DTYPE,
    progress_update_interval=PROGRESS_UPDATE_INTERVAL,
)

confusion, per_class_rows, test_metrics = save_test_data(
    test_result,
    test_loader.dataset,
    test_loader.dataset.classes,
    DATA_OUTPUT_DIR,
    best_checkpoint["epoch"],
    best_checkpoint["val_accuracy"],
)

plot_confusion_matrix(
    confusion,
    test_loader.dataset.classes,
    FIGURE_OUTPUT_DIR / "confusion_matrix.png",
)
plot_per_class_accuracy(
    per_class_rows,
    FIGURE_OUTPUT_DIR / "per_class_accuracy.png",
)
plot_training_and_test_summary(
    history,
    test_metrics,
    FIGURE_OUTPUT_DIR / "training_and_test_summary.png",
)

print(f"最佳验证 epoch：{best_checkpoint['epoch']}")
print(f"最佳验证准确率：{best_checkpoint['val_accuracy']:.2%}")
print(f"最终测试准确率：{test_result['accuracy']:.2%}")


In [ ]:
from IPython.display import Image, display

for figure_name in (
    "training_history.png",
    "confusion_matrix.png",
    "per_class_accuracy.png",
    "training_and_test_summary.png",
):
    display(Image(filename=FIGURE_OUTPUT_DIR / figure_name))
